# Purpose

Prove the department certification contract is bounded, deterministic, and complete.

## Lemma Mapping

- L65 — M365 Department Certification v1

## Expected Results

- All 10 departments have valid packs and pass certification.
- Persona counts match persona registry.
- Sum of department personas equals 39.

In [ ]:
from pathlib import Path
import yaml

repo_root = Path.cwd()
contract = yaml.safe_load(
    (repo_root / 'registry' / 'department_certification_v1.yaml').read_text(encoding='utf-8')
)
registry = yaml.safe_load(
    (repo_root / 'registry' / 'persona_registry_v2.yaml').read_text(encoding='utf-8')
)

assert contract['contract']['id'] == 'department-certification'
assert len(contract['certification_phases']) == 4
assert set(contract['governance_rules'].keys()) == {
    'fail_closed', 'audit_completeness', 'bounded_claims',
    'determinism', 'exhaustive_coverage',
}

# Count personas per department from registry
dept_counts = {}
for p in registry['personas'].values():
    d = p['department']
    dept_counts[d] = dept_counts.get(d, 0) + 1

cert_status = contract['department_certification_status']
total_personas = 0
for dept_id, entry in cert_status.items():
    pack_path = repo_root / 'registry' / f'department_pack_{dept_id.replace("-", "_")}_v1.yaml'
    assert pack_path.exists(), f'missing pack: {pack_path}'
    pack = yaml.safe_load(pack_path.read_text(encoding='utf-8'))
    assert len(pack.get('personas', {})) == entry['persona_count']
    assert entry['persona_count'] == dept_counts.get(dept_id, 0)
    assert entry['workflow_family_count'] == len(pack.get('workflow_families', []))
    assert entry['workflow_family_count'] > 0
    assert entry['certification_verdict'] == 'certified'
    total_personas += entry['persona_count']

assert total_personas == contract['kpis']['total_department_personas']
assert len(cert_status) == contract['kpis']['total_departments']